# ML repair loop

This notebook exercises Notebook Lens as an agent-facing interface for a compact ML loop:

- profile a synthetic classification dataset
- train a baseline logistic model
- hit and inspect a failed hyperparameter attempt
- repair the failed cell in place
- clean-rerun the notebook and inspect artifacts
- refresh the narrative after the evidence is stable

The run keeps scratch files under `tmp/agent_rounds/20260615_round9/ml_repair_loop/` and artifacts under `artifacts/agent_rounds/20260615_round9/ml_repair_loop/`.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path
from statistics import mean, pstdev

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from samples.ml_tasks.common import (  # noqa: E402
    ARTIFACT_DIR,
    FEATURES,
    display_html,
    generate_classification_rows,
    save_json,
    split_rows,
    write_csv,
)

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

rows = generate_classification_rows(n=720, seed=47, drift=0.15)
train_rows, test_rows = split_rows(rows, train_frac=0.72, seed=17)


def describe(values: list[float]) -> dict[str, float]:
    return {
        "mean": mean(values),
        "pstdev": pstdev(values),
        "min": min(values),
        "max": max(values),
    }


feature_profile = {
    feature: describe([float(row[feature]) for row in rows]) for feature in FEATURES
}
label_rate = mean(float(row["label"]) for row in rows)
split_profile = {
    "all_rows": len(rows),
    "train_rows": len(train_rows),
    "test_rows": len(test_rows),
    "label_rate": label_rate,
    "train_label_rate": mean(float(row["label"]) for row in train_rows),
    "test_label_rate": mean(float(row["label"]) for row in test_rows),
}
profile = {
    "dataset": "synthetic retention risk",
    "seed": 47,
    "drift": 0.15,
    "split": split_profile,
    "features": feature_profile,
}

profile_path = save_json(ARTIFACT_DIR / "dataset_profile.json", profile)
summary_path = write_csv(
    ARTIFACT_DIR / "profile_summary.csv",
    [
        {"feature": feature, **{key: round(value, 6) for key, value in stats.items()}}
        for feature, stats in feature_profile.items()
    ],
    ["feature", "mean", "pstdev", "min", "max"],
)

print(f"artifact_dir: {ARTIFACT_DIR}")
print(f"profile_artifact: {profile_path}")
print(f"profile_csv: {summary_path}")
for key, value in split_profile.items():
    print(f"{key}: {value:.4f}" if isinstance(value, float) else f"{key}: {value}")

rows_html = "\n".join(
    "<tr>"
    f"<td>{feature}</td>"
    f"<td>{stats['mean']:.3f}</td>"
    f"<td>{stats['pstdev']:.3f}</td>"
    f"<td>{stats['min']:.3f}</td>"
    f"<td>{stats['max']:.3f}</td>"
    "</tr>"
    for feature, stats in feature_profile.items()
)
display_html(
    "<h3>Dataset Profile</h3>"
    "<table>"
    "<tr><th>feature</th><th>mean</th><th>pstdev</th><th>min</th><th>max</th></tr>"
    f"{rows_html}"
    "</table>"
)


artifact_dir: artifacts/agent_rounds/20260615_round9/ml_repair_loop
profile_artifact: artifacts/agent_rounds/20260615_round9/ml_repair_loop/dataset_profile.json
profile_csv: artifacts/agent_rounds/20260615_round9/ml_repair_loop/profile_summary.csv
all_rows: 720
train_rows: 518
test_rows: 202
label_rate: 0.4222
train_label_rate: 0.4286
test_label_rate: 0.4059


feature,mean,pstdev,min,max
usage,0.042,1.017,-3.676,3.226
tenure,-0.058,0.997,-3.031,3.829
support_load,0.102,1.015,-2.897,3.432
nps,-0.063,0.967,-2.884,2.712
seat_ratio,-0.051,0.983,-2.759,2.692


In [2]:
from __future__ import annotations

from samples.ml_tasks.common import (  # noqa: F821
    ARTIFACT_DIR,
    classification_metrics,
    metrics_table,
    save_json,
    score_rows,
    train_logistic,
    write_csv,
)

baseline_model = train_logistic(train_rows, lr=0.05, epochs=110, l2=0.001)  # noqa: F821
baseline_scored = score_rows(test_rows, baseline_model)  # noqa: F821
baseline_metrics = classification_metrics(baseline_scored)

baseline_model_path = save_json(ARTIFACT_DIR / "baseline_model.json", baseline_model)
baseline_metrics_path = save_json(ARTIFACT_DIR / "baseline_metrics.json", baseline_metrics)
baseline_predictions_path = write_csv(
    ARTIFACT_DIR / "baseline_predictions.csv",
    [
        {
            "id": row["id"],
            "label": row["label"],
            "prob": round(float(row["prob"]), 6),
            "pred": row["pred"],
        }
        for row in baseline_scored
    ],
    ["id", "label", "prob", "pred"],
)

print(f"baseline_model_artifact: {baseline_model_path}")
print(f"baseline_metrics_artifact: {baseline_metrics_path}")
print(f"baseline_predictions_artifact: {baseline_predictions_path}")
print("baseline_history:")
for point in baseline_model["history"]:
    print(f"- epoch {point['epoch']:>3}: loss={point['loss']:.4f}")
print("baseline_test_metrics:")
for key, value in baseline_metrics.items():
    print(f"- {key}: {value:.4f}" if isinstance(value, float) else f"- {key}: {value}")

metrics_table("Baseline Test Metrics", baseline_metrics)


baseline_model_artifact: artifacts/agent_rounds/20260615_round9/ml_repair_loop/baseline_model.json
baseline_metrics_artifact: artifacts/agent_rounds/20260615_round9/ml_repair_loop/baseline_metrics.json
baseline_predictions_artifact: artifacts/agent_rounds/20260615_round9/ml_repair_loop/baseline_predictions.csv
baseline_history:
- epoch   1: loss=0.6931
- epoch  18: loss=0.6226
- epoch  36: loss=0.5740
- epoch  54: loss=0.5414
- epoch  72: loss=0.5187
- epoch  90: loss=0.5021
- epoch 108: loss=0.4897
- epoch 110: loss=0.4886
baseline_test_metrics:
- tp: 58
- fp: 14
- tn: 106
- fn: 24
- precision: 0.8056
- recall: 0.7073
- accuracy: 0.8119
- f1: 0.7532
- brier: 0.1522


tp,58
fp,14
tn,106
fn,24
precision,0.8056
recall,0.7073
accuracy,0.8119
f1,0.7532
brier,0.1522


In [3]:
from __future__ import annotations

from samples.ml_tasks.common import (  # noqa: F821
    ARTIFACT_DIR,
    classification_metrics,
    display_html,
    save_json,
    score_rows,
    split_rows,
    train_logistic,
    write_csv,
)

failed_marker = ARTIFACT_DIR / "failed_hparam_attempt.json"
if failed_marker.exists():
    failed_marker.unlink()
    print(f"removed_failure_marker: {failed_marker}")

tune_train_rows, valid_rows = split_rows(train_rows, train_frac=0.78, seed=23)  # noqa: F821
candidate_grid = [
    {"lr": 0.04, "l2": 0.001, "epochs": 80},
    {"lr": 0.08, "l2": 0.01, "epochs": 120},
    {"lr": 0.12, "l2": 0.001, "epochs": 140},
    {"lr": 0.06, "l2": 0.0, "epochs": 140},
]

results = []
for candidate in candidate_grid:
    model = train_logistic(
        tune_train_rows,
        lr=float(candidate["lr"]),
        l2=float(candidate["l2"]),
        epochs=int(candidate["epochs"]),
    )
    metrics = classification_metrics(score_rows(valid_rows, model))
    result = {**candidate, **metrics}
    results.append(result)
    print(
        f"lr={candidate['lr']:.3f} l2={candidate['l2']:.3f} "
        f"epochs={candidate['epochs']} f1={metrics['f1']:.4f} "
        f"accuracy={metrics['accuracy']:.4f} brier={metrics['brier']:.4f}"
    )

best = max(results, key=lambda row: (row["f1"], row["accuracy"], -row["brier"]))
best_model = train_logistic(
    train_rows,  # noqa: F821
    lr=float(best["lr"]),
    l2=float(best["l2"]),
    epochs=int(best["epochs"]),
)
best_scored = score_rows(test_rows, best_model)  # noqa: F821
best_metrics = classification_metrics(best_scored)
comparison_summary = {
    "baseline": baseline_metrics,  # noqa: F821
    "best_validation_config": {
        "lr": best["lr"],
        "l2": best["l2"],
        "epochs": best["epochs"],
        "validation_f1": best["f1"],
        "validation_accuracy": best["accuracy"],
        "validation_brier": best["brier"],
    },
    "repaired_test": best_metrics,
    "test_f1_delta": best_metrics["f1"] - baseline_metrics["f1"],  # noqa: F821
    "test_brier_delta": best_metrics["brier"] - baseline_metrics["brier"],  # noqa: F821
}

sweep_json_path = save_json(
    ARTIFACT_DIR / "sweep_results.json",
    {"results": results, "best_validation_config": comparison_summary["best_validation_config"]},
)
sweep_csv_path = write_csv(
    ARTIFACT_DIR / "sweep_results.csv",
    [
        {
            key: round(value, 6) if isinstance(value, float) else value
            for key, value in result.items()
        }
        for result in sorted(results, key=lambda row: row["f1"], reverse=True)
    ],
    ["lr", "l2", "epochs", "tp", "fp", "tn", "fn", "precision", "recall", "accuracy", "f1", "brier"],
)
best_model_path = save_json(ARTIFACT_DIR / "best_model.json", best_model)
best_metrics_path = save_json(ARTIFACT_DIR / "best_model_test_metrics.json", best_metrics)
comparison_path = save_json(ARTIFACT_DIR / "comparison_summary.json", comparison_summary)

print(f"sweep_json_artifact: {sweep_json_path}")
print(f"sweep_csv_artifact: {sweep_csv_path}")
print(f"best_model_artifact: {best_model_path}")
print(f"best_metrics_artifact: {best_metrics_path}")
print(f"comparison_artifact: {comparison_path}")
print("best_validation_config:")
for key, value in comparison_summary["best_validation_config"].items():
    print(f"- {key}: {value:.4f}" if isinstance(value, float) else f"- {key}: {value}")
print("repaired_test_metrics:")
for key, value in best_metrics.items():
    print(f"- {key}: {value:.4f}" if isinstance(value, float) else f"- {key}: {value}")
print(f"test_f1_delta: {comparison_summary['test_f1_delta']:.4f}")
print(f"test_brier_delta: {comparison_summary['test_brier_delta']:.4f}")

rows_html = "\n".join(
    "<tr>"
    f"<td>{row['lr']}</td><td>{row['l2']}</td><td>{row['epochs']}</td>"
    f"<td>{row['f1']:.3f}</td><td>{row['accuracy']:.3f}</td><td>{row['brier']:.3f}</td>"
    "</tr>"
    for row in sorted(results, key=lambda item: item["f1"], reverse=True)
)
display_html(
    "<h3>Repaired Hyperparameter Sweep</h3>"
    "<table>"
    "<tr><th>lr</th><th>l2</th><th>epochs</th><th>f1</th><th>accuracy</th><th>brier</th></tr>"
    f"{rows_html}"
    "</table>"
)


lr=0.040 l2=0.001 epochs=80 f1=0.7347 accuracy=0.7719 brier=0.1817
lr=0.080 l2=0.010 epochs=120 f1=0.7475 accuracy=0.7807 brier=0.1617
lr=0.120 l2=0.001 epochs=140 f1=0.7475 accuracy=0.7807 brier=0.1567
lr=0.060 l2=0.000 epochs=140 f1=0.7475 accuracy=0.7807 brier=0.1624
sweep_json_artifact: artifacts/agent_rounds/20260615_round9/ml_repair_loop/sweep_results.json
sweep_csv_artifact: artifacts/agent_rounds/20260615_round9/ml_repair_loop/sweep_results.csv
best_model_artifact: artifacts/agent_rounds/20260615_round9/ml_repair_loop/best_model.json
best_metrics_artifact: artifacts/agent_rounds/20260615_round9/ml_repair_loop/best_model_test_metrics.json
comparison_artifact: artifacts/agent_rounds/20260615_round9/ml_repair_loop/comparison_summary.json
best_validation_config:
- lr: 0.1200
- l2: 0.0010
- epochs: 140
- validation_f1: 0.7475
- validation_accuracy: 0.7807
- validation_brier: 0.1567
repaired_test_metrics:
- tp: 58
- fp: 14
- tn: 106
- fn: 24
- precision: 0.8056
- recall: 0.7073
- acc

lr,l2,epochs,f1,accuracy,brier
0.08,0.01,120,0.747,0.781,0.162
0.12,0.001,140,0.747,0.781,0.157
0.06,0.0,140,0.747,0.781,0.162
0.04,0.001,80,0.735,0.772,0.182


In [4]:
from __future__ import annotations

import hashlib
import json

from samples.ml_tasks.common import ARTIFACT_DIR, display_html, save_json  # noqa: F821

required_artifacts = {
    "dataset_profile.json",
    "profile_summary.csv",
    "baseline_model.json",
    "baseline_metrics.json",
    "baseline_predictions.csv",
    "sweep_results.json",
    "sweep_results.csv",
    "best_model.json",
    "best_model_test_metrics.json",
    "comparison_summary.json",
}

records = []
for path in sorted(ARTIFACT_DIR.iterdir()):
    if path.name == "artifact_inventory.json":
        continue
    payload = path.read_bytes()
    records.append(
        {
            "name": path.name,
            "bytes": len(payload),
            "sha256_12": hashlib.sha256(payload).hexdigest()[:12],
        }
    )

present = {record["name"] for record in records}
missing = sorted(required_artifacts - present)
if missing:
    raise AssertionError(f"missing expected artifacts: {missing}")
if (ARTIFACT_DIR / "failed_hparam_attempt.json").exists():
    raise AssertionError("failed_hparam_attempt.json should not survive the repaired clean run")

comparison = json.loads((ARTIFACT_DIR / "comparison_summary.json").read_text(encoding="utf-8"))
inventory = {
    "artifact_dir": str(ARTIFACT_DIR),
    "file_count_without_inventory": len(records),
    "required_artifacts_present": sorted(required_artifacts),
    "files": records,
    "test_f1_delta": comparison["test_f1_delta"],
    "test_brier_delta": comparison["test_brier_delta"],
}
inventory_path = save_json(ARTIFACT_DIR / "artifact_inventory.json", inventory)

print(f"artifact_inventory: {inventory_path}")
print(f"file_count_without_inventory: {len(records)}")
print(f"test_f1_delta: {comparison['test_f1_delta']:.4f}")
print(f"test_brier_delta: {comparison['test_brier_delta']:.4f}")
for record in records:
    print(f"- {record['name']}: {record['bytes']} bytes sha256={record['sha256_12']}")

rows_html = "\n".join(
    "<tr>"
    f"<td>{record['name']}</td>"
    f"<td>{record['bytes']}</td>"
    f"<td>{record['sha256_12']}</td>"
    "</tr>"
    for record in records
)
display_html(
    "<h3>Artifact Inventory</h3>"
    "<table><tr><th>file</th><th>bytes</th><th>sha256</th></tr>"
    f"{rows_html}</table>"
)


artifact_inventory: artifacts/agent_rounds/20260615_round9/ml_repair_loop/artifact_inventory.json
file_count_without_inventory: 10
test_f1_delta: 0.0000
test_brier_delta: -0.0139
- baseline_metrics.json: 212 bytes sha256=22bd6405c91d
- baseline_model.json: 1381 bytes sha256=0f8d8a45f666
- baseline_predictions.csv: 4239 bytes sha256=375446491932
- best_model.json: 1383 bytes sha256=8ec131df1131
- best_model_test_metrics.json: 211 bytes sha256=91cb90e5a52a
- comparison_summary.json: 789 bytes sha256=a5db72c06104
- dataset_profile.json: 1107 bytes sha256=d7fccb16be2d
- profile_summary.csv: 262 bytes sha256=e2d79e1039c8
- sweep_results.csv: 344 bytes sha256=f636fb77a1b9
- sweep_results.json: 1490 bytes sha256=ceb07ca6d86f


file,bytes,sha256
baseline_metrics.json,212,22bd6405c91d
baseline_model.json,1381,0f8d8a45f666
baseline_predictions.csv,4239,375446491932
best_model.json,1383,8ec131df1131
best_model_test_metrics.json,211,91cb90e5a52a
comparison_summary.json,789,a5db72c06104
dataset_profile.json,1107,d7fccb16be2d
profile_summary.csv,262,e2d79e1039c8
sweep_results.csv,344,f636fb77a1b9
sweep_results.json,1490,ceb07ca6d86f


## Findings

The clean repair loop is stable after the failed hyperparameter attempt was replaced in place.

- Data profile: 720 rows, 518 train rows, 202 test rows, and overall label rate 0.4222.
- Baseline test metrics: F1 0.7532, accuracy 0.8119, Brier 0.1522.
- Repaired sweep winner: `lr=0.12`, `l2=0.001`, `epochs=140`, validation F1 0.7475, validation Brier 0.1567.
- Repaired test metrics: F1 0.7532, accuracy 0.8119, Brier 0.1382.
- Repair impact: test F1 delta 0.0000, Brier delta -0.0139.
- Artifact inspection confirmed 10 required artifacts plus `artifact_inventory.json`; `failed_hparam_attempt.json` was removed by the repair.

The repair did not improve the thresholded classification counts on the held-out split, but it did improve probability calibration enough to lower Brier loss.
